In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")

# Data
DATA_PATH = os.getenv("DATA_PATH")
CHECKPOINT_PATH = os.getenv("CHECKPOINT_PATH")
EXPERIMENT_PATH = os.getenv("EXPERIMENT_PATH")

# Transformações
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [ ]:
# Configuração
BATCH_SIZE = os.getenv("BATCH_SIZE")

# Datasets
train_dataset = datasets.MNIST(DATA_PATH, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(DATA_PATH, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Modelo simples (MLP ou CNN)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 10)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 64 * 5 * 5)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# Configurações
EPOCHS = os.getenv("EPOCHS")
LR = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Treino
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

    # Validação
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    print(f"Accuracy: {100 * correct / total:.2f}%")

In [ ]:
# Salvar checkpoint
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
filepath = CHECKPOINT_PATH + "/mnist_cnn.pth"
torch.save(model.state_dict(), filepath)
print(f"Modelo salvo em {filepath}")